# 05 — SPY 90-Day Forecast (Prophet)

**Goal:** Forecast the S&P 500 ETF (SPY) 90 business days into the future, overlay confirmed anomaly markers on the chart, and evaluate against a held-out test set.

## Why Prophet?

Facebook's **Prophet** is designed for business time series:
- Handles multiple seasonalities (yearly, weekly) automatically
- Robust to missing data and holiday effects
- `changepoint_prior_scale` controls how aggressively the trend adapts — 0.05 is moderately flexible
- Produces 95% confidence bands that visually communicate forecast uncertainty

We evaluate on the **last 6 months** of actual data (out-of-sample) before forecasting 90 days forward.

In [1]:
# ── 1. Imports ─────────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import duckdb
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from pathlib import Path

try:
    from prophet import Prophet
    print('Prophet imported OK')
except ImportError:
    raise ImportError('prophet not found. Run: pip install prophet')

DATA_DIR = Path('data')
OUT_DIR  = Path('outputs')
OUT_DIR.mkdir(exist_ok=True)
print('Imports OK')

Prophet imported OK
Imports OK


In [2]:
# ── 2. Load data ───────────────────────────────────────────────────────────
df_market  = pd.read_csv(DATA_DIR / 'market_data.csv',     parse_dates=['Date'])
df_anomaly = pd.read_csv(DATA_DIR / 'anomaly_results.csv', parse_dates=['Date'])

# Filter SPY
spy = df_market[df_market['Ticker'] == 'SPY'][['Date','Close']].copy()
spy = spy.sort_values('Date').reset_index(drop=True)
print(f'SPY rows: {len(spy)}  |  {spy.Date.min().date()} → {spy.Date.max().date()}')

# SPY confirmed anomalies
spy_anom = df_anomaly[
    (df_anomaly['Ticker'] == 'SPY') & (df_anomaly['IsConfirmed'] == 1)
][['Date','Close']].copy()
print(f'SPY confirmed anomalies: {len(spy_anom)}')

SPY rows: 2071  |  2018-01-02 → 2026-03-30
SPY confirmed anomalies: 14


## Train / Test Split

We hold out the **last 6 months** of actual SPY data as the test set. The model is trained on everything before that, then evaluated on the test set before forecasting further.

In [3]:
# ── 3. Train / test split ──────────────────────────────────────────────────
cutoff = spy['Date'].max() - pd.DateOffset(months=6)
train  = spy[spy['Date'] < cutoff].rename(columns={'Date':'ds','Close':'y'})
test   = spy[spy['Date'] >= cutoff].rename(columns={'Date':'ds','Close':'y'})

print(f'Train: {len(train)} rows  ({train.ds.min().date()} → {train.ds.max().date()})')
print(f'Test : {len(test)}  rows  ({test.ds.min().date()}  → {test.ds.max().date()})')

Train: 1946 rows  (2018-01-02 → 2025-09-29)
Test : 125  rows  (2025-09-30  → 2026-03-30)


## Fit Prophet

Key parameters:
- `changepoint_prior_scale=0.05` — moderate flexibility (lower = more rigid trend)
- `yearly_seasonality=True` — captures January effect, end-of-year moves
- `weekly_seasonality=True` — captures Monday/Friday patterns
- `interval_width=0.95` — 95% credible interval for the forecast band

In [4]:
# ── 4. Fit Prophet ─────────────────────────────────────────────────────────
model = Prophet(
    changepoint_prior_scale=0.05,
    yearly_seasonality=True,
    weekly_seasonality=True,
    interval_width=0.95
)
model.fit(train)
print('Prophet model fitted.')

16:23:54 - cmdstanpy - INFO - Chain [1] start processing


16:23:55 - cmdstanpy - INFO - Chain [1] done processing


Prophet model fitted.


In [5]:
# ── 5. Forecast 90 business days forward ──────────────────────────────────
last_date    = spy['Date'].max()
future_bdays = pd.bdate_range(start=last_date + pd.Timedelta(days=1), periods=90)
future_df    = pd.DataFrame({'ds': future_bdays})

# Combine train + test + future into one prediction frame
full_future = pd.concat([train[['ds']], test[['ds']], future_df], ignore_index=True)
forecast    = model.predict(full_future)

print(f'Forecast rows: {len(forecast)}')
print(forecast[['ds','yhat','yhat_lower','yhat_upper']].tail(5).to_string(index=False))

Forecast rows: 2161
        ds       yhat  yhat_lower  yhat_upper
2026-07-28 702.371029  592.805240  829.376371
2026-07-29 702.506881  596.789539  821.902095
2026-07-30 702.656267  587.122348  826.693239
2026-07-31 702.880766  587.888193  821.114036
2026-08-03 703.389948  592.185069  825.308870


## Evaluate on Test Set

**MAE** (Mean Absolute Error) is in dollar terms — easy to interpret.  
**MAPE** (Mean Absolute Percentage Error) normalises to percentage — comparable across price levels.

In [6]:
# ── 6. Evaluate on test set ────────────────────────────────────────────────
test_forecast = forecast[forecast['ds'].isin(test['ds'])][['ds','yhat']].copy()
eval_df = test.merge(test_forecast, on='ds')

mae  = np.mean(np.abs(eval_df['y'] - eval_df['yhat']))
mape = np.mean(np.abs((eval_df['y'] - eval_df['yhat']) / eval_df['y'])) * 100

print(f'Test set evaluation ({len(eval_df)} trading days):')
print(f'  MAE  = ${mae:.2f}')
print(f'  MAPE = {mape:.2f}%')
if mape < 5:
    print('  → Excellent fit (MAPE < 5%)')
elif mape < 10:
    print('  → Good fit (MAPE 5–10%)')
else:
    print('  → Moderate fit (MAPE > 10% — expected for long-horizon equity forecasting)')

Test set evaluation (125 trading days):
  MAE  = $26.30
  MAPE = 3.88%
  → Excellent fit (MAPE < 5%)


## DuckDB — Forecast Horizon Summary

SQL CTE pattern: compute row numbers in the inner query, filter in the outer. Avoids the error of using window functions directly in WHERE.

In [7]:
# ── 7. DuckDB — forecast horizon summary ─────────────────────────────────
forecast_future = forecast[forecast['ds'] > last_date][['ds','yhat','yhat_lower','yhat_upper']].copy()
forecast_future = forecast_future.rename(columns={'ds':'Date'})

con = duckdb.connect()
con.register('fc', forecast_future)

horizon = con.execute("""
    WITH ranked AS (
        SELECT
            ROW_NUMBER() OVER (ORDER BY Date)          AS biz_day,
            Date,
            ROUND(yhat, 2)        AS forecast,
            ROUND(yhat_lower, 2)  AS lower_95,
            ROUND(yhat_upper, 2)  AS upper_95,
            FIRST_VALUE(yhat) OVER (ORDER BY Date)     AS first_yhat,
            yhat
        FROM fc
    )
    SELECT
        biz_day, Date, forecast, lower_95, upper_95,
        ROUND((yhat - first_yhat) / first_yhat * 100, 2) AS chg_from_today_pct
    FROM ranked
    WHERE biz_day IN (1, 5, 10, 20, 30, 45, 60, 75, 90)
""").df()

print('=== Forecast at Key Horizons (DuckDB) ===')
print(horizon.to_string(index=False))

=== Forecast at Key Horizons (DuckDB) ===
 biz_day       Date  forecast  lower_95  upper_95  chg_from_today_pct
       1 2026-03-31    656.85    600.30    715.13                0.00
       5 2026-04-06    658.42    597.43    723.08                0.24
      10 2026-04-13    659.49    601.05    727.62                0.40
      20 2026-04-27    662.34    595.96    733.50                0.84
      30 2026-05-11    666.89    595.33    750.48                1.53
      45 2026-06-01    677.49    597.09    766.86                3.14
      60 2026-06-22    684.45    592.53    784.62                4.20
      75 2026-07-13    697.34    596.26    806.79                6.16
      90 2026-08-03    703.39    592.19    825.31                7.08


## Final Chart — Forecast + Actuals + Anomaly Markers

The chart overlays:
- **Grey**: Historical SPY price (training period)
- **Dark blue**: Actual SPY price in the test window
- **Orange dashed**: Prophet forecast line
- **Light orange band**: 95% confidence interval
- **Red open circles**: Confirmed anomaly dates (from notebook 04)

In [8]:
# ── 8. Forecast chart (improved styling) ─────────────────────────────────
plt.rcParams.update({
    'figure.dpi'       : 150,
    'font.family'      : 'DejaVu Sans',
    'axes.spines.top'  : False,
    'axes.spines.right': False,
    'axes.grid'        : True,
    'grid.alpha'       : 0.2,
    'grid.linestyle'   : ':',
})

fig, ax = plt.subplots(figsize=(16, 7))

# Confidence band
fc_plot = forecast.set_index('ds').sort_index()
ax.fill_between(fc_plot.index, fc_plot['yhat_lower'], fc_plot['yhat_upper'],
                alpha=0.20, color='#FF9800', label='95% Confidence Band', zorder=1)

# Forecast line
ax.plot(fc_plot.index, fc_plot['yhat'],
        color='#E65100', linewidth=1.8, linestyle='--',
        label='Prophet Forecast', zorder=3)

# Historical actuals (grey)
train_actual = spy[spy['Date'] < cutoff]
ax.plot(train_actual['Date'], train_actual['Close'],
        color='#BDBDBD', linewidth=1.2, label='SPY Historical', zorder=2)

# Test actuals (dark blue — the held-out evaluation window)
test_actual = spy[spy['Date'] >= cutoff]
ax.plot(test_actual['Date'], test_actual['Close'],
        color='#1565C0', linewidth=2.2, label='SPY Actual (test set)', zorder=4)

# Confirmed anomaly markers
if len(spy_anom) > 0:
    ax.scatter(spy_anom['Date'], spy_anom['Close'],
               facecolors='none', edgecolors='red',
               s=100, linewidths=2.0, zorder=6, label='Confirmed Anomaly')

# Reference verticals
ylo, yhi = ax.get_ylim()
ax.axvline(cutoff,    color='#1565C0', linestyle=':', linewidth=1.3, alpha=0.8)
ax.axvline(last_date, color='#2E7D32', linestyle=':', linewidth=1.3, alpha=0.8)

ax.text(cutoff    + pd.Timedelta(days=4), yhi * 0.98,
        'Train/Test\nsplit', color='#1565C0', fontsize=8.5, va='top',
        bbox=dict(boxstyle='round,pad=0.25', facecolor='white', alpha=0.8))
ax.text(last_date + pd.Timedelta(days=4), yhi * 0.91,
        'Forecast\nstart', color='#2E7D32', fontsize=8.5, va='top',
        bbox=dict(boxstyle='round,pad=0.25', facecolor='white', alpha=0.8))

ax.set_title(
    f'SPY — 90-Day Prophet Forecast\n'
    f'Test MAE = ${mae:.2f}   |   Test MAPE = {mape:.2f}%   |   '
    f'Forecast horizon: {future_bdays[0].date()} → {future_bdays[-1].date()}',
    fontsize=13, fontweight='bold', pad=14
)
ax.set_xlabel('Year', fontsize=11)
ax.set_ylabel('SPY Close Price ($)', fontsize=11)
ax.legend(loc='upper left', fontsize=9, framealpha=0.95, ncol=2)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax.xaxis.set_major_locator(mdates.YearLocator())

plt.tight_layout()
save_path = OUT_DIR / 'spy_forecast.png'
plt.savefig(save_path, dpi=150, bbox_inches='tight')
print(f'Saved -> {save_path}')
plt.show()

Saved -> outputs\spy_forecast.png


In [9]:
# ── 9. Final project summary ──────────────────────────────────────────────
print('\n' + '='*62)
print('PROJECT COMPLETE — Output File Inventory')
print('='*62)
all_files = list(Path('data').glob('*')) + list(Path('outputs').glob('*'))
for f in sorted(all_files):
    print(f'  {str(f):<45}  {f.stat().st_size/1024:>7.1f} KB')
print('='*62)
print(f'\n  Forecast MAE  = ${mae:.2f}')
print(f'  Forecast MAPE = {mape:.2f}%')


PROJECT COMPLETE — Output File Inventory
  data\anomaly_results.csv                        4760.4 KB
  data\market_data.csv                            1943.3 KB
  data\market_data_indicators.csv                 3816.9 KB
  outputs\anomaly_AAPL.png                         148.2 KB
  outputs\anomaly_AMZN.png                         165.6 KB
  outputs\anomaly_GLD.png                          137.4 KB
  outputs\anomaly_JPM.png                          142.1 KB
  outputs\anomaly_SPY.png                          146.3 KB
  outputs\anomaly_XOM.png                          144.6 KB
  outputs\correlation_heatmap.png                  104.8 KB
  outputs\normalised_price.png                     367.1 KB
  outputs\rolling_volatility.png                   352.7 KB
  outputs\spy_forecast.png                         260.7 KB
  outputs\spy_monthly_heatmap.png                  151.9 KB

  Forecast MAE  = $26.30
  Forecast MAPE = 3.88%
